# 01 Eksplorasi Data OULAD

Notebook ini mengeksplorasi `data/processed/dataset.csv` hasil ekstraksi OULAD.
Label `at_risk=1` berarti hasil akhir `Fail` atau `Withdrawn`. Fitur hanya memakai jendela hari 0 sampai 41.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'training' else Path.cwd()
DATA = ROOT / 'data' / 'processed'
df = pd.read_csv(DATA / 'dataset.csv')
with (DATA / 'extract_info.json').open(encoding='utf-8') as f:
    info = json.load(f)

print(f'Ukuran dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print('Kolom:', ', '.join(df.columns))
df.head()

## 1. Ringkasan ekstraksi dan label

In [ ]:
print('Pendaftaran awal      :', f"{info['pendaftaran_awal']:,}")
print('Dikeluarkan sebelum mulai:', f"{info['excluded_early_unregistration']:,}")
print('Pendaftaran akhir     :', f"{info['pendaftaran_akhir']:,}")
print('Label positif         :', f"{info['label_positif']:,}")
print('Proporsi berisiko     :', f"{info['label_positif_rate']:.1%}")

label_counts = df['at_risk'].value_counts().sort_index()
label_table = pd.DataFrame({
    'jumlah': label_counts,
    'proporsi': (label_counts / len(df)).round(4),
}, index=['tidak berisiko', 'berisiko'])
label_table

## 2. Kualitas data dan nilai kosong

Nilai kosong dibiarkan pada tahap ini. Imputasi dilakukan di dalam pipeline model dan hanya memakai data latih.

In [ ]:
feature_columns = [
    'aktivitas_7h', 'aktivitas_14h', 'hari_sejak_aktif',
    'rata_nilai_tugas', 'tren_nilai',
    'jumlah_tugas_dikumpulkan', 'rasio_tugas_selesai',
]
missing = df[feature_columns].isna().sum().rename('jumlah_missing').to_frame()
missing['proporsi'] = (missing['jumlah_missing'] / len(df)).round(4)
missing

## 3. Distribusi fitur numerik

In [ ]:
df[feature_columns].describe().T.round(2)

In [ ]:
axes = df[feature_columns].hist(figsize=(13, 9), bins=30, color='#1D5C7A', edgecolor='white')
plt.suptitle('Distribusi fitur input', y=1.02)
plt.tight_layout()
plt.show()

## 4. Hubungan fitur dengan label

Perbandingan median membantu melihat sinyal awal tanpa mengubah data atau memilih parameter model. Pemilihan model tetap dilakukan oleh pipeline training.

In [ ]:
grouped = df.groupby('at_risk')[feature_columns].median().T
grouped.columns = ['tidak_berisiko', 'berisiko']
grouped['selisih_berisiko_minus_tidak'] = grouped['berisiko'] - grouped['tidak_berisiko']
grouped.round(2)

In [ ]:
plot_columns = ['aktivitas_7h', 'aktivitas_14h', 'rata_nilai_tugas', 'rasio_tugas_selesai']
fig, axes = plt.subplots(1, len(plot_columns), figsize=(14, 3.5))
for ax, column in zip(axes, plot_columns):
    df.boxplot(column=column, by='at_risk', ax=ax, grid=False)
    ax.set_title(column)
    ax.set_xlabel('at_risk')
    ax.set_ylabel('nilai')
plt.suptitle('Perbandingan fitur berdasarkan label', y=1.04)
plt.tight_layout()
plt.show()

## 5. Kesimpulan EDA

- Dataset berisi satu baris per pendaftaran mata kuliah.
- Proporsi label berisiko perlu dilaporkan karena memengaruhi interpretasi accuracy.
- Nilai kosong terutama muncul pada `rata_nilai_tugas`, `tren_nilai`, dan `rasio_tugas_selesai`; pipeline model menangani imputasi.
- `tren_nilai` dipertahankan berdasarkan uji banding per fold pada `03b_ablasi_tren.py`.
- EDA ini bersifat deskriptif; keputusan model dan fitur tetap mengikuti pipeline training.